# When a zero is a lie

*Question, Intuition, Math, Code, Assumptions, How it breaks*

One of my eleven requirements is **discipline**: cards and fouls, negated so a
clean player scores above a dirty one. It eliminates more players than any other
requirement in the definition.

It is also built on a column the source did not always record. And
`groupby().sum()` will turn a column of nothing into a column of zeros without a
word of complaint. A zero in a fouls column says something very specific, and
false: *he fouled nobody.*

Here is the part I did not expect. This turns out to be **completely harmless in
most of the data and genuinely damaging in a little of it**, and which is which
has nothing to do with how much is missing.

## 1. Question

A column the gate depends on is missing for part of the data. What do I put
there, and how much does the answer actually matter?

## 2. Intuition

There are three answers, and the usual way people choose between them is to ask
how *much* is missing. I think that is the wrong question.

**Put a zero.** This is what you get by not deciding. It claims the player fouled
nobody.

**Put the group's average.** Now the player scores neither well nor badly, which
sounds fair. But an average carries no variance, so injecting it narrows the
spread of whatever group it lands in.

**Drop the term.** Discipline falls back to cards alone. It measures something
coarser, but it measures it the same way for everyone.

The question that actually decides the damage is not how much is missing but
**where**. Specifically, whether the hole lines up with the groups you normalise
inside. Every score in this project is z-scored within its own
`(league, season)`. A hole that fills one of those groups completely behaves
nothing like a hole scattered across one.

## 3. Math

The textbook taxonomy first. **MCAR** means absence is independent of everything,
so dropping and imputing are both safe. **MAR** means absence depends on columns
you observed, so conditional imputation is defensible. **MNAR** means absence
depends on the value itself or on something you never measured, and nothing is
safe.

Fouls here are not MCAR. Absence is predicted almost perfectly by
`(league, season)`, which technically makes them MAR given a variable I hold.

But the taxonomy is not what decides the damage, and this is the bit I want to
show you. Take a requirement $x$ z-scored within group $g$:

$$z_{ig} = \frac{x_{ig} - \bar{x}_g}{\sigma_g}$$

If every member of $g$ is missing, and every one gets filled with the same
constant, that constant goes into $\bar{x}_g$ and cancels straight back out. The
fill **cannot** change any $z$ in that group. It adds nothing to the spread and
nothing to the mean difference.

If instead a handful inside $g$ get filled while the rest carry real values,
those few have been placed at a specific point on a scale everybody else earned.
Now the fill is a claim, and it is being compared against measurements.

**Same fill, same column, same fraction missing, opposite consequences.**

## 4. Code

### Where the hole actually is

In [1]:
import numpy as np
import pandas as pd

from gambeta import level, needs

seasons = pd.read_parquet("../data/sample/player_season_scored.parquet")
seasons["year"] = 2000 + seasons["season"].str[:2].astype(int)

covered = seasons.groupby(["league", "year"])["fouls"].apply(lambda s: s.notna().mean())
covered.to_frame("share").pivot_table(index="league", columns="year", values="share").loc[
    :, :2007
].round(2)

year,2000,2001,2002,2003,2004,2005,2006,2007
league,,,,,,,,
ENG-Premier League,1.0,0.99,1.00,1.00,0.99,1.00,0.99,1.0
ESP-La Liga,1.0,1.00,1.00,0.99,0.99,1.00,1.00,1.0
FRA-Ligue 1,0.0,0.00,0.00,0.00,0.00,0.00,0.98,1.0
GER-Bundesliga,0.0,0.00,0.00,0.00,0.00,0.00,1.00,1.0
ITA-Serie A,1.0,1.00,0.99,0.99,0.99,0.99,0.99,1.0


That is not an era ramp, and it is not record-keeping gradually improving.

**England, Spain and Italy have fouls from 2000 onwards.** France and Germany
have none at all until 2006, and then have them completely. Twelve league-seasons
of exactly nothing, sitting next to sixty-odd that are exactly complete.

I want to flag what I nearly did here. An average taken across all five leagues
reports "about 60% coverage in the early years", and that describes a dataset
which does not exist. There is no league-season anywhere at 60%.

In [2]:
share = seasons.groupby(["league", "year"])["fouls"].transform(lambda s: s.notna().mean())
regime = pd.cut(
    share,
    [-0.01, 0.001, 0.999, 1.01],
    labels=["wholly missing", "partly covered", "fully covered"],
)

summary = (
    seasons.assign(regime=regime)
    .groupby("regime", observed=True)
    .agg(
        rows=("fouls", "size"),
        missing=("fouls", lambda s: int(s.isna().sum())),
    )
)
summary["share missing"] = (summary["missing"] / summary["rows"]).map("{:.1%}".format)
summary

,rows,missing,share missing
regime,,,
wholly missing,3508,3508,100.0%
partly covered,6433,48,0.7%
fully covered,29936,0,0.0%


Three regimes, and only the middle one is a problem.

### A hole that fills the whole group is free

Take one of the twelve wholly-missing league-seasons and score discipline two
ways: with zeros, and with the term dropped entirely.

In [3]:
group = seasons[(seasons["league"] == "GER-Bundesliga") & (seasons["year"] == 2002)].copy()
per90 = group["minutes"] / 90

group["with_zeros"] = -(group["red"] + group["second_yellow"] + group["fouls"].fillna(0) / per90)
group["term_dropped"] = -(group["red"] + group["second_yellow"])

scored = level.zscore(group, ["with_zeros", "term_dropped"])
same = np.allclose(scored["with_zeros_z"], scored["term_dropped_z"])

print(f"{len(group)} players, {group['fouls'].isna().sum()} missing a fouls figure")
print(f"identical z-scores either way: {same}")
print("\nThe fill was constant across the whole group, so it cancelled.")

287 players, 287 missing a fouls figure
identical z-scores either way: True

The fill was constant across the whole group, so it cancelled.


Nothing to argue about there. The zero was a lie, and the lie made no
difference, because every player in the group told exactly the same one.

### A hole scattered inside the group is not free

Now take a league-season that is *almost* complete. This is the middle regime,
where a handful of players lack a figure and everyone else has one.

In [4]:
partial = seasons[(share > 0.001) & (share < 0.999)]
worst = partial.groupby(["league", "year"])["fouls"].apply(lambda s: s.isna().sum()).idxmax()
g = seasons[(seasons["league"] == worst[0]) & (seasons["year"] == worst[1])].copy()
rate = g["fouls"] / (g["minutes"] / 90)

measured = rate.dropna()
print(f"{worst[0]} {worst[1]}: {len(g)} players, {int(rate.isna().sum())} without a fouls figure\n")
print("a zero-filled player is recorded at  : 0.00 fouls per 90")
print(f"measured players strictly worse      : {100 * (measured > 0).mean():.0f}%")
print(f"measured players genuinely also at 0 : {int((measured == 0).sum())}")
print("\nThe fill invents no record better than anyone's. It places every unrecorded")
print("player at the cleanest end of a scale the others had to earn.")

FRA-Ligue 1 2006: 312 players, 5 without a fouls figure

a zero-filled player is recorded at  : 0.00 fouls per 90
measured players strictly worse      : 92%
measured players genuinely also at 0 : 25

The fill invents no record better than anyone's. It places every unrecorded
player at the cleanest end of a scale the others had to earn.


That is the real cost, and I want to be precise about its size rather than
dramatic: **610 rows out of 65,069** across the whole dataset. Small. Not zero,
and not anywhere I would have thought to look.

### The guard I added, and what it actually does

Below a coverage floor the fouls term gets dropped. Above it, fouls count
normally.

In [5]:
import inspect

print(inspect.getsource(needs._foul_rate))
print(f"FOUL_COVERAGE = {needs.FOUL_COVERAGE}")

def _foul_rate(df: pd.DataFrame, minutes: pd.Series) -> np.ndarray:
    """Fouls per 90, dropped in league-seasons the source barely recorded."""
    if "fouls" not in df.columns:
        return np.zeros(len(df), dtype=float)
    recorded = df["fouls"].notna()
    covered = recorded.groupby([df["league"], df["season"]]).transform("mean")
    usable = (recorded & (covered >= FOUL_COVERAGE)).to_numpy()
    return np.where(usable, _rate(df["fouls"].fillna(0.0), minutes), 0.0)

FOUL_COVERAGE = 0.8


In [6]:
clean = seasons.rename(columns={"year": "_year"})
gated = needs._foul_rate(clean, clean["minutes"])
naive = needs._rate(clean["fouls"].fillna(0.0), clean["minutes"])

print(f"rows where the guard changes the answer: {int((~np.isclose(gated, naive)).sum())}")

rows where the guard changes the answer: 0


**Zero rows.**

The guard is aimed at league-seasons below 80% coverage, and on this data every
single such group sits at 0%, which is exactly the regime where the fill already
cancels out. The 610 rows that genuinely are distorted live in groups at 95 to
99% coverage, comfortably above my floor, and the guard sails straight over them.

It is a correct-looking rule that carefully protects the case which was already
safe.

## 5. Assumptions

1. **Absence is a property of the source, not the player.** A missing fouls
   figure means FBref published none. It is never read as "he committed none",
   which is precisely what the zero claims.
2. **Cards alone are still a discipline signal.** Coarser, yes, but reds and
   second yellows are recorded throughout, so the wholly-missing groups still
   rank players on something real.
3. **The grain of normalisation is `(league, season)`.** Every argument in this
   chapter depends on that. Change the grain and which holes are harmless changes
   with it.

## 6. How it breaks

**The guard above is this chapter's own worked example, and I am leaving it in
for that reason.**

I wrote it from a description of the problem that was entirely true: "fouls are
patchy in the early era, so drop the term where coverage is thin." Every word of
that sentence is correct. It is also aimed at the wrong regime, and the only
thing that showed me was running the comparison in the cell above and getting
zero back.

No test would have caught it. There is no bug in it. It computes exactly what it
says, its output is sensible, and the number it produces is right. It simply does
nothing, because the assumption underneath it, that thin coverage and damaging
coverage are the same thing, was never checked against the data.

That is the same failure mode as the four defects in the validation chapter,
committed by me, while writing the fix for one of them.

**What the honest fix would be.** For the middle regime, a missing player should
score the group's mean rather than zero. Neutral instead of best-in-class. The
variance-shrinkage objection to mean-imputation is real, but at 2.4% of a group
it is negligible, and that objection is exactly what I used to reject the option
that turns out to be right.

**What no fix repairs.** Discipline in Bundesliga 2002 ranks players on cards. In
La Liga 2002 it ranks them on cards and fouls. Both get z-scored onto the same
scale, the gate treats them as the same requirement, and they are not measuring
the same thing. Three Ballon d'Or winners, Nedvěd, Figo and Rodri, fail my gate
on discipline alone.

That is a limitation of the source. It belongs out in the open rather than
smoothed over with a plausible-looking fill.